# Pipeline de Détection de Fausses Nouvelles (Branche Style)

## Vue d'ensemble du notebook

**Objectif:** Orchestre un pipeline complet de 5 phases pour la détection des fausses nouvelles basée sur l'analyse de style:
1. **Ingestion de données** (fusion de 5 sources)
2. **Extraction de caractéristiques stylométriques** (20+ métriques)
3. **Découpage des données** (train/val/test stratifié 60-20-20)
4. **Entraînement RoBERTa** (fine-tuning sur tâche de classification)
5. **Évaluation et comparaison d'ensembles**

**Environnement:** Google Colab (requiert accès au lecteur)

---

## Architecture du Pipeline

```
Source de Données (5 sources)
   ↓
Phase 1: Ingestion de Données
├─ Décompression des fichiers
└─ Fusion des 5 sources (LIAR, Twitter, UoVictoria)
   ↓
Phase 2: Extraction de Caractéristiques de Style
├─ Normalisation du texte (masquage d'entités)
├─ Extraction de 20+ métriques stylométriques
├─ Analyse de sentiment (VADER)
└─ Métriques de lisibilité (Flesch-Kincaid)
   ↓
Phase 3: Découpage et Stratification
├─ Split stratifié 60-20-20
├─ Block A: RoBERTa (données d'entraînement)
├─ Block B: Ensemble (validation)
└─ Block C: Test final
   ↓
Phase 4: Entraînement RoBERTa
├─ Fine-tuning distilroberta-base
├─ Augmentation des données avec probabilités
└─ Enregistrement du modèle fine-tuné
   ↓
Phase 5: Évaluation et Résultats
├─ Comparaison RandomForest vs XGBoost
├─ Analyse d'importance des caractéristiques
└─ Rapports de performance (Accuracy, F1, ROC-AUC)
```

---

## Phases d'exécution

### Phase 1: Ingestion de Données
**Durée:** < 1 minute
- Décompression des fichiers
- Fusion des 5 sources (LIAR, Twitter, UoVictoria)
- Sortie: `complete_train.csv`

### Phase 2: Extraction de Caractéristiques de Style
**Durée:** 5-15 minutes
- 20+ métriques stylométriques (longueur phrases, complexité lexicale, sentiment, etc.)
- Normalisation de texte (masquage d'URL, mentions, nombres)
- Sortie: `complete_train.csv` augmenté

### Phase 3: Découpage et Stratification
**Durée:** 30 secondes
- Split stratifié 60-20-20 (maintient ratio Faux/Vrai)
- Block A (60%): RoBERTa fine-tuning
- Block B (20%): Validation ensemble
- Block C (20%): Test final

### Phase 4: Fine-tuning RoBERTa
**Durée:** 20-40 min (GPU), 2-4h (CPU)
- Modèle: distilroberta-base
- Learning rate: 2e-5, Batch: 8, Epochs: 2
- Augmentation Block B/C avec probabilités RoBERTa
- Sortie: Modèle fine-tuné + fichiers augmentés

### Phase 5: Comparaison d'Ensembles
**Durée:** 10-20 minutes
- RandomForestClassifier: 15 configurations, 5-fold CV
- XGBClassifier: 15 configurations, 5-fold CV
- Sélection: F1 score primaire, log loss tiebreaker
- Sorties: best_model.pkl, rapports, feature_importance.png

---

## Hyperparamètres clés

| Paramètre | Valeur | Justification |
|-----------|--------|---------------|
| Learning rate | 2e-5 | Standard fine-tuning BERT |
| Batch size | 8 | Optimisé pour GPU standard |
| Epochs | 2 | Prévient overfitting |
| Max tokens | 512 | Standard DistilBERT |
| Claim detection threshold | 0.2 | Conservateur; réduit faux négatifs |
| Entailment threshold | 0.70 | Classification SUPPORTÉE |

---

## Temps d'exécution total

- **GPU (CUDA/MPS):** 1-2 heures
- **CPU:** 8-12 heures (fortement recommandé GPU)
- **Espace disque requis:** 5-10 GB

---

## Sorties attendues

| Fichier | Source | Utilisation |
|---------|--------|------------|
| `complete_train.csv` | Phase 1-2 | Données + caractéristiques |
| `block_A_roberta_train.csv` | Phase 3 | Entraînement RoBERTa |
| `block_B_train_WITH_PROB.csv` | Phase 4 | Block B + probabilités |
| `block_C_final_test_WITH_PROB.csv` | Phase 4 | Block C + probabilités |
| `roberta_fine_tunned/` | Phase 4 | Modèle sauvegardé |
| `results/best_model.pkl` | Phase 5 | Meilleur classifieur |
| `results/report_random_forest.txt` | Phase 5 | Métriques RF |
| `results/report_xgboost.txt` | Phase 5 | Métriques XGB |

---

## Exécution du pipeline

Exécutez chaque cellule **dans l'ordre** pour orchestrer toutes les 5 phases.

## First step

In [1]:
#!python unzip.py
print("Unzipping files...")

Unzipping files...


## Style based detection

### 1. Data preparation

If it has already been run, there is no need to run it again (but you can if you have 40 spare minutes)

In [2]:
%cd data/
!python data_extraction.py
%cd ..

%cd style_branch
!python feature_extraction.py

!python print_features.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/data
Loading datasets...

DONE! Fusion of dataset have been saved to: dataset.csv

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch
Loading data from ../data/dataset.csv...

Initializing NLP engine (spaCy + TextBlob/VADER)...
Loading linguistic models...
Extracting Style Metrics: 100%|███████████| 61673/61673 [27:24<00:00, 37.50it/s]

DONE! Features have been saved to: ../data/complete_train.csv
NEWS ANALYSIS:

Raw text:
ALERT! You must absolutely read this: https://www.google.com. The government is hiding 10,000 terrible and monstrous things! Wake up immediately @user! #news 😡

Loading linguistic models...
Normalized text:
ALERT! You must absolutely read this: [URL] The government is hiding [NB] terrible and monstrous things! Wake up immediately [MENTION]! #news 😡

Style vector (FEATURES):

has_hashtags  

### 2. Fine tuning RoBERTa
Same as the previous cell, you don't need to run it, it can be very very very long because of your GPU (if you have it)
(MacBook Pro M4 with 20 GPU core (48 Go unified memory) -> 41 minutes)

In [3]:
%cd style_branch
!python split_data.py

!python fine_tunning.py

%run test_fine_tuned.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch
Loading the complete dataset (Style + Text)...

Data distribution:
Block A (RoBERTa Train) : 37003 rows
Block B (Training) : 12335 rows
Block C (Final Test)    : 12335 rows

Sanity Check - Class Distribution (Fake=1, True=0):
Block A (RoBERTa): 
label
0    0.5
1    0.5
Name: proportion, dtype: float64
Block B (XGBoost): 
label
0    0.5
1    0.5
Name: proportion, dtype: float64
Block C (Test):    
label
0    0.5
1    0.5
Name: proportion, dtype: float64

Splitting complete and files saved!
Nvidia GPU detected! Training will be hardware-accelerated.
Loading CSV files...
Word tokenization in progress... This may take a minute.
Loading weights: 100%|█████████████████████| 101/101 [00:00<00:00, 17479.87it/s]
RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Analyzed sentence: 'The Federal Reserve announced a 0.5% increase in interest rates today.'
TRUE NEWS probability: 60.88%
FAKE NEWS probability: 39.12%

Analyzed sentence: 'BREAKING: Alien spaceship found hidden under the White House! The government is lying to us!!!'
TRUE NEWS probability: 15.24%
FAKE NEWS probability: 84.76%

Analyzed sentence: ''
TRUE NEWS probability: 75.87%
FAKE NEWS probability: 24.13%

Analyzed sentence: ''
TRUE NEWS probability: 75.87%
FAKE NEWS probability: 24.13%

Analyzed sentence: 'Enter'
TRUE NEWS probability: 89.03%
FAKE NEWS probability: 10.97%
/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news


### 3. Random Forest x XGBoost competition
If warnings appears about killing child process it is because you are on MacOs and it auto kill child process and the Python processus find any child process already kild by native MacOs processsus so it's just a warning and no action required. Others os aren't concerned.

In [4]:
%cd style_branch
!python model_comp.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch
Loading data (Style + RoBERTa Semantics)...

Configuration: 15 combinations tested per model (Cross-Validation x5).
Optimizing Random Forest...

Training Random Forest: 100%|███████████████████| 75/75 [00:12<00:00,  5.89it/s]
Finished in 0.21 min. Best parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_depth': 10, 'class_weight': 'balanced'}
Optimizing XGBoost...

Training XGBoost: 100%|█████████████████████████| 75/75 [00:07<00:00, 10.11it/s]
Finished in 0.12 min. Best parameters: {'subsample': 0.8, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.9}

Model                | Accuracy             | ROC-AUC (Quality)    | F1 Score             | Log Loss             |
------------------------------------------------------------------------------------------------------------------
Random Forest        |               92.55% |           

In [5]:
%cd style_branch
!python result_roberta.py
%cd ..

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news/style_branch
Évaluation RoBERTa: 100%|█████████████████████| 771/771 [01:41<00:00,  7.58it/s]

RÉSULTATS DE ROBERTA SEUL (BASELINE)
Accuracy  : 92.44%
F1 Score  : 92.21%
ROC-AUC   : 98.52%
Log Loss  : 15.30%
--------------------------------------------------

Rapport détaillé :

              precision    recall  f1-score   support

           0       0.90      0.95      0.93      6173
           1       0.95      0.90      0.92      6162

    accuracy                           0.92     12335
   macro avg       0.93      0.92      0.92     12335
weighted avg       0.93      0.92      0.92     12335

/home/juul/Documents/IFT714 Traitement des LN/Projet/Detection_fake_news
